In [1]:
import os, warnings, json, itertools
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import re
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import ttest_ind, ks_2samp
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, Tuple
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.stats import ttest_ind, ks_2samp

# Nature Communications 常用色板（色盲友好、简洁）
PALETTE = ["#2166AC", "#67A9CF", "#D1E5F0", "#F4A582", "#B2182B"]
sns.set_theme(style="whitegrid", palette=PALETTE, font_scale=1.1)

In [2]:
# ==== 通用工具 ====
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score
from scipy.stats import spearmanr, kendalltau
from numpy.linalg import svd, norm

plt.rcParams['figure.dpi'] = 150

def expected_calibration_error(y_true, y_prob, n_bins=10):
    """返回 ECE、每个 bin 的预测均值、观测命中率、计数"""
    y_true = np.asarray(y_true); y_prob = np.asarray(y_prob)
    bins = np.linspace(0, 1, n_bins+1)
    idx = np.digitize(y_prob, bins) - 1
    ece = 0.0
    bin_pred, bin_true, counts = [], [], []
    for b in range(n_bins):
        m = idx == b
        if m.sum() == 0:
            bin_pred.append(np.nan); bin_true.append(np.nan); counts.append(0); continue
        p_hat = y_prob[m].mean()
        p_obs = y_true[m].mean()
        ece += (m.sum()/len(y_true)) * abs(p_hat - p_obs)
        bin_pred.append(p_hat); bin_true.append(p_obs); counts.append(int(m.sum()))
    return ece, np.array(bin_pred), np.array(bin_true), np.array(counts), bins

def youden_threshold(y_true, y_prob):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    j = tpr - fpr
    return thr[np.argmax(j)]

def tpr_fpr_at(y_true, y_prob, thr):
    y_hat = (y_prob >= thr).astype(int)
    tp = np.sum((y_true==1)&(y_hat==1))
    fn = np.sum((y_true==1)&(y_hat==0))
    fp = np.sum((y_true==0)&(y_hat==1))
    tn = np.sum((y_true==0)&(y_hat==0))
    tpr = tp/(tp+fn+1e-9); fpr = fp/(fp+tn+1e-9)
    return tpr, fpr

def linear_recoverability_auc(X, y_bin):
    """线性探针预测敏感属性的 AUC；返回 AUC 与 |AUC-0.5|"""
    m = ~np.isnan(y_bin)
    if m.sum() == 0: return np.nan, np.nan
    clf = LogisticRegression(max_iter=200).fit(X[m], y_bin[m])
    prob = clf.predict_proba(X[m])[:,1]
    auc = roc_auc_score(y_bin[m], prob)
    return auc, abs(auc-0.5)

def concept_erasure_nullspace(Z, probes):
    """
    probes: dict{name: (coef_vector, bias)}，如线性分类器权重
    用多个方向的列空间做 SVD，再把 Z 投影到其正交补空间，实现线性去混杂（共形擦除）
    """
    W = []
    for _, (w,_) in probes.items():
        if w is None: continue
        w = w.reshape(-1,1)
        if norm(w) > 0: W.append(w/norm(w))
    if not W: return Z.copy()
    W = np.hstack(W)   # d x k
    U, S, Vt = svd(W, full_matrices=False)
    # 正交补空间投影矩阵 P = I - U U^T
    P = np.eye(Z.shape[1]) - U @ U.T
    return Z @ P

def fit_linear_probe(X, y_bin):
    """返回 (coef, intercept, AUC, |AUC-0.5|)"""
    m = ~np.isnan(y_bin)
    if m.sum() == 0: return None, None, np.nan, np.nan
    clf = LogisticRegression(max_iter=200).fit(X[m], y_bin[m])
    prob = clf.predict_proba(X[m])[:,1]
    auc = roc_auc_score(y_bin[m], prob)
    return clf.coef_.ravel(), clf.intercept_[0], auc, abs(auc-0.5)

# ==== 读取与准备数据 ====
FILE = "data_processed.xlsx"
doc  = pd.read_excel(FILE, sheet_name="doc_clean")
mach = pd.read_excel(FILE, sheet_name="machine_clean")

# 统一列名小写
doc.columns  = doc.columns.str.strip().str.lower()
mach.columns = mach.columns.str.strip().str.lower()
df = doc.merge(mach, on=["date", "name"], suffixes=("_doc", "_ai"))
min_count = df["认知功能_doc"].value_counts().min()
df = (df.groupby("认知功能_doc", group_keys=False)
        .apply(lambda x: x.sample(min_count, random_state=42))
        .reset_index(drop=True))
# 假设df已包含 q1_ai~q30_ai, q1_doc~q30_doc, age, gender, edu, language, name, date
q_doc = [col for col in df.columns if col.endswith('_doc')][:30]
q_ai  = [col for col in df.columns if col.endswith('_ai')][:30]

pop_vars = ["age", "edu", "gender", "language"]
df = df.dropna(subset=q_doc)
df = df.dropna(subset=q_ai)
df[q_doc] = df[q_doc].astype(int)
df["total_doc"] = df[q_doc].sum(axis=1)
df[q_ai] = df[q_ai].astype(int)
df["total_ai"] = df[q_ai].sum(axis=1)

# 类别均衡（你 notebook 使用过）
if "认知功能_doc" in df.columns:
    min_count = df["认知功能_doc"].value_counts().min()
    df = (df.groupby("认知功能_doc", group_keys=False)
             .apply(lambda x: x.sample(min_count, random_state=42))
             .reset_index(drop=True))

# 题目列
q_doc = [c for c in df.columns if c.endswith("_doc")][:30]
q_ai  = [c for c in df.columns if c.endswith("_ai")][:30]

# 计算总分/差异/分箱
df[q_doc] = df[q_doc].astype(int)
df[q_ai]  = df[q_ai].astype(int)
df["total_doc"] = df[q_doc].sum(axis=1)
df["total_ai"]  = df[q_ai].sum(axis=1)
df["delta"] = df["total_ai"] - df["total_doc"]
df["abs_delta"] = df["delta"].abs()
df = df.sort_values("abs_delta").reset_index(drop=True)
df["abs_bin"] = (df.index / len(df) * 10).astype(int)

# 医生标签二值化（兼容“认知正常/认知障碍”或 0/1）
if df["认知功能_doc"].dtype == object:
    df["y_doc"] = df["认知功能_doc"].map({"认知正常":0, "认知障碍":1})
else:
    df["y_doc"] = df["认知功能_doc"].astype(int)

# 人口学分组（与 notebook 保持一致）
df["Gender"] = df["gender"].astype(str).str.contains("男").astype(int)
df["Lan"]    = df["language"].astype(str).str.contains("普通").astype(int)  # 1=普通话, 0=方言
df["Age"]    = df["age"]
df["Edu"]    = df["edu"]

df["age_bin"]    = np.where(df["Age"]>65, ">65", "≤65")
df["edu_bin"]    = np.where(df["Edu"]>9, ">9 Years", "≤9 Years")
df["gender_bin"] = np.where(df["Gender"]==1, "Male", "Female")
df["lan_bin"]    = np.where(df["Lan"]==1, "Mandarin", "Dialect")

# 预测概率列（若你已有 prob 列，可替换这里）
if "prob_ai" in df.columns:
    df["p_ai"] = df["prob_ai"].astype(float)
else:
    # 用 total_ai/30 作为概率近似（单调一致，足够绘制校准/EO）
    df["p_ai"] = df["total_ai"]/30.0

# 可选的站点列（若无则自动跳过 LOSO）
has_site = "site" in df.columns


In [3]:
# ====== VAE + latent spectrum (all labels in English) ======
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score
from scipy.stats import spearmanr, kendalltau

import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class VAE(nn.Module):
    def __init__(self, input_dim=30, latent_dim=8, hidden=128):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU())
        self.mu = nn.Linear(hidden, latent_dim)
        self.logvar = nn.Linear(hidden, latent_dim)
        self.dec = nn.Sequential(nn.Linear(latent_dim, hidden), nn.ReLU(),
                                 nn.Linear(hidden, input_dim), nn.Sigmoid())
    def encode(self, x):
        h = self.enc(x)
        return self.mu(h), self.logvar(h)
    def reparam(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparam(mu, logvar)
        recon = self.dec(z)
        return recon, mu, logvar

def vae_loss(recon, x, mu, logvar, beta=0.2):
    rec = nn.functional.mse_loss(recon, x, reduction='mean')
    kld = -0.5*torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return rec + beta*kld

def fit_vae_and_get_Z(X, seed=42, latent_dim=8, epochs=120, batch=64, beta=0.2):
    torch.manual_seed(seed); np.random.seed(seed)
    scaler = StandardScaler().fit(X); Xs = scaler.transform(X).astype(np.float32)
    dl = DataLoader(TensorDataset(torch.tensor(Xs)), batch_size=batch, shuffle=True)
    vae = VAE(input_dim=X.shape[1], latent_dim=latent_dim).to(device)
    opt = torch.optim.Adam(vae.parameters(), lr=1e-3)
    for ep in range(epochs):
        vae.train()
        for (xb,) in dl:
            xb = xb.to(device)
            recon, mu, logvar = vae(xb)
            loss = vae_loss(recon, xb, mu, logvar, beta)
            opt.zero_grad(); loss.backward(); opt.step()
    vae.eval()
    with torch.no_grad():
        mu, _ = vae.encode(torch.tensor(Xs).to(device))
        Z = mu.cpu().numpy()
    Z = StandardScaler().fit_transform(Z)
    return Z, vae




In [4]:
FILE = "data_processed.xlsx"
doc  = pd.read_excel(FILE, sheet_name="doc_clean")
mach = pd.read_excel(FILE, sheet_name="machine_clean")

# 统一列名小写
doc.columns  = doc.columns.str.strip().str.lower()
mach.columns = mach.columns.str.strip().str.lower()
df = doc.merge(mach, on=["date", "name"], suffixes=("_doc", "_ai"))
min_count = df["认知功能_doc"].value_counts().min()
df = (df.groupby("认知功能_doc", group_keys=False)
        .apply(lambda x: x.sample(min_count, random_state=42))
        .reset_index(drop=True))
# 假设df已包含 q1_ai~q30_ai, q1_doc~q30_doc, age, gender, edu, language, name, date
q_doc = [col for col in df.columns if col.endswith('_doc')][:30]
q_ai  = [col for col in df.columns if col.endswith('_ai')][:30]

pop_vars = ["age", "edu", "gender", "language"]
df = df.dropna(subset=q_doc)
df = df.dropna(subset=q_ai)
df[q_doc] = df[q_doc].astype(int)
df["total_doc"] = df[q_doc].sum(axis=1)
df[q_ai] = df[q_ai].astype(int)
df["total_ai"] = df[q_ai].sum(axis=1)

# 总分 & 差异
df["delta"]     = df[q_ai].sum(axis=1) - df[q_doc].sum(axis=1)
df["abs_delta"] = df["delta"].abs()

# 分箱
df = df.sort_values("abs_delta").reset_index(drop=True)
df["abs_bin"] = (df.index / len(df) * 10).astype(int)   # 0~9

# 人口学 one-hot
df["Gender"] = df["gender"].astype(str).str.contains("男").astype(int)
df["Lan"]  = df["language"].astype(str).str.contains("普通").astype(int)
df["Age"]  = df["age"]
df["Edu"]  = df["edu"]
# 假定 df 已有 abs_bin, gender_m, age, edu, lang_cn 等列
df["age_group"] = (df["age"] > 65).astype(int)    # 1=老年组，0=≤65
df["edu_group"] = (df["edu"] > 9).astype(int)     # 1=高教育，0=低教育
df["gender_group"] = df["gender"].astype(str).str.contains("男").astype(int)
df["lan_group"]  = df["language"].astype(str).str.contains("普通").astype(int)

In [5]:
def get_features(df, idx, use_vars=[]):
    feat = q_ai.copy()  # 一定要保证q_ai是全局定义、顺序不变
    # 增加人口学变量
    if "Age" in use_vars:
        feat.append("Age")
    if "Edu" in use_vars:
        feat.append("Edu")
    if "Gender" in use_vars:
        feat.append("Gender")
    if "Lan" in use_vars:
        feat.append("Lan")
    return df.loc[idx, feat].astype(float).values


def run_repel_bin(df, bin_id, use_vars=[], n_models=5, disturb_rate=0.1, random_state=42):
    rng = np.random.RandomState(random_state)
    idx_high = np.where(df["abs_bin"]==bin_id)[0]
    idx_low  = np.where(df["abs_bin"]==0)[0]
    n_pos = len(idx_high)
    n_neg = len(idx_low)
    if n_pos < 5 or n_neg < n_pos:  # 低差异池比高差异区还少则跳过
        return None

    # 严格采样n_pos个低差异池样本
    X_high = get_features(df, idx_high, use_vars)
    X_low_idx = rng.choice(idx_low, size=n_pos, replace=False)
    X_low  = get_features(df, X_low_idx, use_vars)
    y_high = np.ones(n_pos)
    y_low  = np.zeros(n_pos)
    results = []
    for m in range(n_models):
        y_high_flip = y_high.copy()
        n_flip = int(len(y_high_flip)*disturb_rate)
        if n_flip > 0:
            flip_idx = rng.choice(len(y_high_flip), n_flip, replace=False)
            y_high_flip[flip_idx] = 1-y_high_flip[flip_idx]
        X = np.vstack([X_high, X_low])
        y = np.hstack([y_high_flip, y_low])
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=rng.randint(1e9))
        clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
        ypred = clf.predict(Xte)
        yprob = clf.predict_proba(Xte)[:,1]
        acc = accuracy_score(yte, ypred)
        f1 = f1_score(yte, ypred)
        auc = roc_auc_score(yte, yprob)
        results.append((acc, f1, auc, yprob))
    preds = np.stack([r[3] for r in results], axis=1)
    entropy = -np.mean(preds*np.log(preds+1e-8)+(1-preds)*np.log(1-preds+1e-8), axis=1)
    var     = np.var(preds, axis=1)
    return {
        "acc_mean": np.mean([r[0] for r in results]),
        "f1_mean" : np.mean([r[1] for r in results]),
        "auc_mean": np.mean([r[2] for r in results]),
        "entropy_mean": np.mean(entropy),
        "var_mean": np.mean(var),
        "pred_entropy": entropy,
        "pred_var": var
    }




In [6]:
# 分特征集组合（空=仅AI题目；单独/全部人口学变量）
var_sets = [[],
            ["Age"],["Edu"],["Gender"],["Lan"],
            ["Age","Gender"],["Age","Gender","Edu"],["Age","Gender","Edu","Lan"]]

results = []
for vars_ in var_sets:
    for b in range(10):
        out = run_repel_bin(df, b, use_vars=vars_)
        if out:
            out.update({"bin":b, "feature_set":"+".join(vars_) or "AI Only"})
            results.append(out)
repel_df = pd.DataFrame(results)


# Second

In [7]:
import warnings; warnings.filterwarnings("ignore")
repel_file = repel_df
entropy_map = repel_df.set_index("bin")["entropy_mean"].to_dict()
df["entropy_mean"] = df["abs_bin"].map(entropy_map)
df["entropy_mean"] = df["entropy_mean"].fillna(df["entropy_mean"].mean())
# 反向归一化权重
entropy = df["entropy_mean"].values
sample_weights = 1 - ((entropy - entropy.min()) / (entropy.ptp() + 1e-8))
sample_weights = 0.5 + 0.5 * sample_weights


In [8]:

from sklearn.preprocessing import MinMaxScaler
q_ai_cols = [col for col in df.columns if col.endswith('_ai')][:30]
X = MinMaxScaler().fit_transform(df[q_ai_cols].astype(float).values)

# 可选去偏：如用INLP见上一轮答复（建议先跑不去偏版）


In [9]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
from sklearn.utils import resample
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ---- 改1：更通用的VAE（MSE重构 + 可调beta + 可调latent_dim） ----
class VAE(nn.Module):
    def __init__(self, input_dim=30, latent_dim=16, hidden_dim=128):
        super().__init__()
        self.enc_fc1 = nn.Linear(input_dim, hidden_dim)
        self.enc_fc2_mu = nn.Linear(hidden_dim, latent_dim)
        self.enc_fc2_logvar = nn.Linear(hidden_dim, latent_dim)
        self.dec_fc1 = nn.Linear(latent_dim, hidden_dim)
        self.dec_fc2 = nn.Linear(hidden_dim, input_dim)

    def encode(self, x):
        h = torch.relu(self.enc_fc1(x))
        return self.enc_fc2_mu(h), self.enc_fc2_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = torch.relu(self.dec_fc1(z))
        return self.dec_fc2(h)  # 不再sigmoid，配合MSE

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

def vae_loss(recon_x, x, mu, logvar, weight=None, beta=0.2):
    # recon: (N, D), x: (N, D)
    mse = ((recon_x - x) ** 2).sum(dim=1)  # per-sample
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)  # per-sample
    if weight is not None:
        # 权重做clipping，防止极端样本主导
        w = torch.clamp(weight, torch.quantile(weight, 0.05), torch.quantile(weight, 0.95))
        loss = torch.mean(w * (mse + beta * kld))
    else:
        loss = torch.mean(mse + beta * kld)
    return loss

X_tensor = torch.tensor(X, dtype=torch.float32)
weights_tensor = torch.tensor(sample_weights, dtype=torch.float32)
dataset = TensorDataset(X_tensor, weights_tensor)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

vae = VAE(input_dim=X.shape[1], latent_dim=2, hidden_dim=64).to(device)
optimizer = torch.optim.Adam(vae.parameters(), lr=0.001)
n_epochs = 200

vae.train()
for epoch in range(n_epochs):
    epoch_loss = 0
    for xb, wb in dataloader:
        xb = xb.to(device)
        wb = wb.to(device)
        recon, mu, logvar = vae(xb)
        loss = vae_loss(recon, xb, mu, logvar, wb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(xb)
    if (epoch+1)%10 == 0:
        print(f"Epoch {epoch+1}, Loss={epoch_loss / len(dataset):.4f}")


Epoch 10, Loss=3.5733
Epoch 20, Loss=3.3974
Epoch 30, Loss=3.2795
Epoch 40, Loss=3.2051
Epoch 50, Loss=3.1853
Epoch 60, Loss=3.1721
Epoch 70, Loss=3.1341
Epoch 80, Loss=3.1338
Epoch 90, Loss=3.1359
Epoch 100, Loss=3.1061
Epoch 110, Loss=3.1066
Epoch 120, Loss=3.1017
Epoch 130, Loss=3.1039
Epoch 140, Loss=3.0930
Epoch 150, Loss=3.0844
Epoch 160, Loss=3.0838
Epoch 170, Loss=3.0787
Epoch 180, Loss=3.0735
Epoch 190, Loss=3.0620
Epoch 200, Loss=3.0723


In [10]:
vae.eval()
with torch.no_grad():
    X_torch = torch.tensor(X, dtype=torch.float32).to(device)
    mus, logvars = vae.encode(X_torch)
    Z = mus.cpu().numpy()

# Generate table

In [11]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score
from scipy.stats import ttest_ind
import torch
import torch.nn as nn
import torch.optim as optim

import pandas as pd
import numpy as np

# 1. 读取Excel
file = 'data_processed.xlsx'
doc  = pd.read_excel(file, sheet_name='doc_clean')
mach = pd.read_excel(file, sheet_name='machine_clean')

# 2. 统一主键列名小写，合并
doc.columns = doc.columns.str.strip().str.lower()
mach.columns = mach.columns.str.strip().str.lower()
df = doc.merge(mach, on=['date','name'], suffixes=('_doc','_ai'))

# 3. 取30道题的列名
q_doc = [col for col in df.columns if col.endswith('_doc')][:30]
q_ai  = [col for col in df.columns if col.endswith('_ai')][:30]
df = df.dropna(subset=q_doc)
df = df.dropna(subset=q_ai)
# 4. 数据类型处理
df[q_doc] = df[q_doc].astype(int)
df[q_ai]  = df[q_ai].astype(int)


# 人口学变量二值化
df['age_bin'] = (df['age'] > 65).astype(int)
df['edu_bin'] = (df['edu'] > 9).astype(int)
df['gender_bin'] = df['gender'].astype(str).str.contains('男').astype(int)
df['lan_bin'] = df['language'].astype(str).str.contains('普通').astype(int)

demo_bin = {
    "Age": "age_bin",
    "Edu": "edu_bin",
    "Gender": "gender_bin",
    "Lan": "lan_bin"
}
label_col = '认知功能_doc'  # 改为你的标签列
# label转0/1
df['label_bin'] = (df[label_col] == "认知障碍").astype(int)

# 不去偏原始组
X_base = MinMaxScaler().fit_transform(df[q_ai].astype(float).values)
y = df['label_bin'].values


In [12]:
from numpy.linalg import norm
from sklearn.linear_model import LogisticRegression
import numpy as np

from numpy.linalg import norm
from sklearn.linear_model import LogisticRegression
import numpy as np

def inlp_project_controlled(X, sens, num_iterations=1):
    """
    通过固定迭代次数来控制去偏强度的INLP。

    参数:
    X (np.array): 输入特征矩阵。
    sens (np.array): 敏感属性的二值化标签。
    num_iterations (int): 固定的投影迭代次数。这个值就是我们的“控制旋钮”。
    """
    # 确保输入是稳定的，防止inplace修改
    X_proj = X.copy()
    
    # P矩阵用于追踪总的投影变换，如果需要可以返回它
    P = np.eye(X.shape[1]) 
    
    # 进行固定次数的迭代
    for i in range(num_iterations):
        # 每次迭代都用当前被修改过的数据集X_proj来训练新的探针
        # 这确保了我们能找到残余空间中最重要的下一个方向
        # 添加random_state保证每次运行结果一致
        clf = LogisticRegression(solver='liblinear', C=0.1, max_iter=1000, random_state=42+i).fit(X_proj, sens)
        w = clf.coef_[0]
        w_norm = norm(w)
        
        # 如果权重向量几乎为零，说明已经没有可分离的信号了，提前中止
        if w_norm < 1e-6:
            # print(f"Converged early at iteration {i+1}")
            break
            
        w = w / w_norm
        
        # 定义当前这一步的投影矩阵（硬投影，alpha=1.0）
        P_w = np.eye(X.shape[1]) - np.outer(w, w)
        
        # 将投影应用到数据上
        X_proj = X_proj @ P_w
        
        # 累积总的投影矩阵
        P = P @ P_w
        
    return X_proj
iteration_map = {
    "Remove-Age": 3,      # 假设 T=4 对 Age 效果最好
    "Remove-Edu": 4,      # 假设 Edu 信号稍弱，需要更强的去偏
    "Remove-Gender": 8,   # Gender 信号最弱，T=2 即可
    "Remove-Lan": 3       # Lan 和 Age 类似
}

X_sets_finetuned = {'No-Deconfound': X_base}
# 2. 循环生成去偏后的数据集，可以尝试不同的alpha值
# 例如，我们用一个相对温和的alpha=0.8
deconfound_alpha = 0.8
for name, col in zip(['Remove-Age','Remove-Edu','Remove-Gender','Remove-Lan'],
                     ['age_bin','edu_bin','gender_bin','lan_bin']):
    T = iteration_map[name]
    X_sets_finetuned[name] = inlp_project_controlled(X_base, df[col].values, num_iterations=T)



In [13]:
from sklearn.utils import resample

def balanced_train_split(X, y, test_size=0.3, random_state=42):
    # 拆分训练和测试
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, stratify=y, random_state=random_state)
    # 拆成正负类
    idx_pos = np.where(y_tr == 1)[0]
    idx_neg = np.where(y_tr == 0)[0]
    n_min = min(len(idx_pos), len(idx_neg))
    # 分别采样
    idx_pos_down = resample(idx_pos, n_samples=n_min, replace=False, random_state=random_state)
    idx_neg_down = resample(idx_neg, n_samples=n_min, replace=False, random_state=random_state)
    idx_balanced = np.concatenate([idx_pos_down, idx_neg_down])
    # 打乱顺序
    np.random.shuffle(idx_balanced)
    X_tr_bal = X_tr[idx_balanced]
    y_tr_bal = y_tr[idx_balanced]
    return X_tr_bal, X_te, y_tr_bal, y_te


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# 可用的下游模型字典
model_dict = {
    "Logistic": LogisticRegression(max_iter=500),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

def run_vae_and_eval_multi(
    X, y,
    sample_weights=None,        # 可选：REPEL不确定性权重（与X同长度）
    epochs=100,
    latent_dim=16,              # 从2提到16，分类更稳；画图时再用UMAP/t-SNE降到2D
    beta=0.2,                   # 降低KL权重，保留更多判别信息
    batch_size=64,
    n_splits=5,
    model_name="Logistic",
    rep="vae",                  # 'vae' | 'pca' | 'none'
    random_state=42
):
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.svm import SVC
    from xgboost import XGBClassifier

    model_dict = {
        "Logistic": LogisticRegression(max_iter=1000, random_state=random_state),
        "RandomForest": RandomForestClassifier(n_estimators=200, random_state=random_state),
        "SVM": SVC(probability=True, random_state=random_state),
        "XGBoost": XGBClassifier(eval_metric='logloss', random_state=random_state)
    }
    model = model_dict[model_name]

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    accs, f1s, aucs, precs, recalls = [], [], [], [], []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        Xtr, Xte = X[train_idx], X[test_idx]
        ytr, yte = y[train_idx], y[test_idx]

        # 下采样（仅训练集）
        idx_pos = np.where(ytr == 1)[0]
        idx_neg = np.where(ytr == 0)[0]
        n_min = min(len(idx_pos), len(idx_neg))
        idx_pos_down = resample(idx_pos, n_samples=n_min, replace=False, random_state=random_state)
        idx_neg_down = resample(idx_neg, n_samples=n_min, replace=False, random_state=random_state)
        idx_balanced = np.concatenate([idx_pos_down, idx_neg_down])
        np.random.shuffle(idx_balanced)
        Xtr_bal, ytr_bal = Xtr[idx_balanced], ytr[idx_balanced]

        # 对应的权重（如果提供）
        w_tr_bal = None
        if sample_weights is not None:
            w_tr_bal = sample_weights[train_idx][idx_balanced]
            # 轻度变换避免过度放大
            w_tr_bal = np.sqrt(np.clip(w_tr_bal, np.percentile(w_tr_bal, 5), np.percentile(w_tr_bal, 95)))

        # 标准化（避免特征量纲影响）
        x_scaler = StandardScaler().fit(Xtr_bal)
        Xtr_bal_sc = x_scaler.transform(Xtr_bal)
        Xte_sc = x_scaler.transform(Xte)

        if rep == "none":
            Ztr, Zte = Xtr_bal_sc, Xte_sc

        elif rep == "pca":
            pca = PCA(n_components=0.95, random_state=random_state)
            Ztr = pca.fit_transform(Xtr_bal_sc)
            Zte = pca.transform(Xte_sc)

        elif rep == "vae":
            # 训练VAE（仅用训练集，避免泄漏）
            vae = VAE(input_dim=Xtr_bal_sc.shape[1], latent_dim=latent_dim, hidden_dim=128).to(device)
            opt = torch.optim.Adam(vae.parameters(), lr=2e-3)

            Xtr_t = torch.tensor(Xtr_bal_sc, dtype=torch.float32).to(device)
            if w_tr_bal is not None:
                w_tr_t = torch.tensor(w_tr_bal, dtype=torch.float32).to(device)
            else:
                w_tr_t = None

            ds = TensorDataset(Xtr_t) if w_tr_t is None else TensorDataset(Xtr_t, w_tr_t)
            dl = DataLoader(ds, batch_size=batch_size, shuffle=True)

            vae.train()
            for ep in range(epochs):
                for batch in dl:
                    if w_tr_t is None:
                        xb, = batch
                        wb = None
                    else:
                        xb, wb = batch
                    recon, mu, logvar = vae(xb)
                    loss = vae_loss(recon, xb, mu, logvar, weight=wb, beta=beta)
                    opt.zero_grad(); loss.backward(); opt.step()

            # 提取潜在表示（用mu），并再做一遍标准化
            vae.eval()
            with torch.no_grad():
                mu_tr, _ = vae.encode(torch.tensor(Xtr_bal_sc, dtype=torch.float32).to(device))
                mu_te, _ = vae.encode(torch.tensor(Xte_sc, dtype=torch.float32).to(device))
                Ztr = mu_tr.cpu().numpy()
                Zte = mu_te.cpu().numpy()

        else:
            raise ValueError("rep must be one of {'vae','pca','none'}")

        # 再对Z做标准化（对SVM/LR更友好）
        z_scaler = StandardScaler().fit(Ztr)
        Ztr_std = z_scaler.transform(Ztr)
        Zte_std = z_scaler.transform(Zte)

        # 下游分类
        model.fit(Ztr_std, ytr_bal)
        y_pred = model.predict(Zte_std)

        # 概率/决策函数
        try:
            y_prob = model.predict_proba(Zte_std)[:, 1]
        except Exception:
            try:
                y_prob = model.decision_function(Zte_std)
            except Exception:
                y_prob = y_pred.astype(float)

        accs.append(accuracy_score(yte, y_pred))
        f1s.append(f1_score(yte, y_pred))
        try:
            aucs.append(roc_auc_score(yte, y_prob))
        except Exception:
            aucs.append(np.nan)
        precs.append(precision_score(yte, y_pred))
        recalls.append(recall_score(yte, y_pred))

    return {
        "auc_list": aucs, "acc_list": accs, "f1_list": f1s,
        "precision_list": precs, "recall_list": recalls,
        "auc": np.nanmean(aucs), "acc": np.nanmean(accs), "f1": np.nanmean(f1s),
        "precision": np.nanmean(precs), "recall": np.nanmean(recalls)
    }


def vae_loss(recon_x, x, mu, logvar, weight=None, beta=0.2):
    # recon: (N, D), x: (N, D)
    mse = ((recon_x - x) ** 2).sum(dim=1)  # per-sample
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)  # per-sample
    if weight is not None:
        # 权重做clipping，防止极端样本主导
        w = torch.clamp(weight, torch.quantile(weight, 0.05), torch.quantile(weight, 0.95))
        loss = torch.mean(w * (mse + beta * kld))
    else:
        loss = torch.mean(mse + beta * kld)
    return loss


In [ ]:
from sklearn.utils import resample
import torch
import numpy as np
import pandas as pd

n_repeat = 10  # 重复次数
model_names = ["Logistic", "RandomForest", "SVM", "XGBoost"]
fair_groups = list(X_sets_finetuned.keys())  # ['No-Removal', ...]
results_list = []

for model_name in model_names:
    for fg in fair_groups:
        aucs, accs, f1s, precs, recalls = [], [], [], [], []
        for repeat in range(n_repeat):
            # 设不同种子，保证每次划分/采样不同
            random_state = 42 + repeat
            metrics = run_vae_and_eval_multi(
                X_sets_finetuned[fg], y, epochs=100, model_name=model_name, n_splits=5, random_state=random_state
            )
            aucs.append(np.mean(metrics['auc_list']))
            accs.append(np.mean(metrics['acc_list']))
            f1s.append(np.mean(metrics['f1_list']))
            precs.append(np.mean(metrics['precision_list']))
            recalls.append(np.mean(metrics['recall_list']))
        results_list.append({
            "model": model_name,
            "feature_group": fg,
            "auc_list": aucs,
            "acc_list": accs,
            "f1_list": f1s,
            "precision_list": precs,
            "recall_list": recalls,
            "auc": np.mean(aucs),
            "acc": np.mean(accs),
            "f1": np.mean(f1s),
            "precision": np.mean(precs),
            "recall": np.mean(recalls),
        })

results_df = pd.DataFrame(results_list)

In [ ]:
from scipy.stats import ttest_ind

results_df["T-value"] = ""
results_df["P-value"] = ""

for model in model_names:
    base_row = results_df[(results_df["model"] == model) & (results_df["feature_group"] == "No-Removal")]
    if len(base_row) == 0:
        continue
    base_auc = base_row.iloc[0]['auc_list']
    for g in ["Remove-Age", "Remove-Edu", "Remove-Gender", "Remove-Lan"]:
        cur_row = results_df[(results_df["model"] == model) & (results_df["feature_group"] == g)]
        if len(cur_row) == 0:
            continue
        cur_auc = cur_row.iloc[0]['auc_list']
        t_stat, p_val = ttest_ind(base_auc, cur_auc, equal_var=False)
        mask = (results_df["model"] == model) & (results_df["feature_group"] == g)
        results_df.loc[mask, "T-value"] = f"{t_stat:.6f}"
        results_df.loc[mask, "P-value"] = f"{p_val:.6f}"


In [ ]:
results = results_df[["model", "feature_group", "auc", "acc", "f1", "precision", "recall", "T-value", "P-value"]]
results

,model,feature_group,auc,acc,f1,precision,recall,T-value,P-value
0,Logistic,No-Deconfound,0.947675,0.877311,0.904701,0.937668,0.874395,,
1,Logistic,Remove-Age,0.877190,0.804588,0.843531,0.904307,0.791095,,
2,Logistic,Remove-Edu,0.763815,0.704961,0.761012,0.826789,0.705717,,
3,Logistic,Remove-Gender,0.918371,0.849155,0.883267,0.912105,0.856636,,
4,Logistic,Remove-Lan,0.899613,0.822536,0.858249,0.917724,0.806795,,
5,RandomForest,No-Deconfound,0.943652,0.866454,0.894383,0.944972,0.849518,,
6,RandomForest,Remove-Age,0.905509,0.816861,0.853262,0.915390,0.799850,,
7,RandomForest,Remove-Edu,0.865744,0.776653,0.821861,0.876906,0.774313,,
8,RandomForest,Remove-Gender,0.920537,0.841791,0.876923,0.910537,0.846324,,
9,RandomForest,Remove-Lan,0.907457,0.819693,0.855203,0.919801,0.799751,,


## 逐一删除特征的对比

In [18]:
import numpy as np
from numpy.linalg import norm
from sklearn.linear_model import LogisticRegression

def inlp_project_multiple_controlled(X, sens_df, num_iterations=1):
    """
    通过固定迭代次数来联合移除多个敏感属性。

    参数:
    X (np.array): 输入特征矩阵。
    sens_df (pd.DataFrame): 包含一个或多个敏感属性列的DataFrame。
    num_iterations (int): 固定的投影迭代次数。
    """
    X_proj = X.copy()
    
    for i in range(num_iterations):
        # 1. 为当前迭代找到所有敏感方向
        w_list = []
        for col in sens_df.columns:
            sens = sens_df[col].values
            clf = LogisticRegression(solver='liblinear', C=0.1, max_iter=1000, random_state=42+i).fit(X_proj, sens)
            w = clf.coef_[0]
            w_norm = norm(w)
            
            if w_norm > 1e-6:
                w_list.append(w / w_norm)
        
        # 如果没有找到任何有效的敏感方向，则提前退出
        if not w_list:
            # print("No more sensitive directions found. Stopping early.")
            break
        
        # 2. 构建敏感子空间的正交基
        W_matrix = np.array(w_list)
        
        # 使用SVD来稳健地找到W_matrix行空间的正交基
        # Vh的行向量就是我们需要的正交基
        _, _, Vh = np.linalg.svd(W_matrix, full_matrices=False)
        
        # Vh的每一行是一个基向量，B是基矩阵
        B = Vh 
        
        # 3. 计算并应用投影矩阵
        # P = I - B^T * B
        projection_matrix = np.eye(X.shape[1]) - B.T @ B
        
        X_proj = X_proj @ projection_matrix
        
    return X_proj


import pandas as pd
import numpy as np
import itertools
from sklearn.preprocessing import MinMaxScaler



# 您设定的单特征去偏强度
iteration_map = {
    "Remove-Age": 3,
    "Remove-Edu": 4,
    "Remove-Gender": 8,
    "Remove-Lan": 3
}

# 基础的人口学变量列名和映射
demo_cols = ['age_bin', 'edu_bin', 'gender_bin', 'lan_bin']
demo_map = {
    'age_bin': 'Age',
    'edu_bin': 'Edu',
    'gender_bin': 'Gender',
    'lan_bin': 'Lan'
}

# 最终将包含所有待测试数据集的字典
# 包含 'No-Deconfound', 'Remove-Age', 'Remove-Age+Gender', etc.
X_sets_all = {'No-Deconfound': X_base}

print("Preparing deconfounded datasets for all combinations...")

# 循环生成1、2、3、4个特征的组合
for k in range(1, len(demo_cols) + 1):
    # 使用itertools.combinations生成所有大小为k的组合
    for combo in itertools.combinations(demo_cols, k):
        # combo 是一个元组, e.g., ('age_bin', 'gender_bin')
        
        # 1. 创建特征组的名称
        feature_group_name = "Remove-" + "+".join([demo_map[c] for c in combo])
        
        # 2. 准备敏感属性DataFrame
        sens_df = df[list(combo)]
        
        # 3. 决定这次联合去偏的迭代次数 (取组合中最大的T值)
        # 这样做可以确保最难去除的属性也能得到充分处理
        max_iterations = 0
        for col in combo:
            # 从单属性的 T 值中找到最大值
            single_feature_name = "Remove-" + demo_map[col]
            if iteration_map[single_feature_name] > max_iterations:
                max_iterations = iteration_map[single_feature_name]

        print(f"Processing: {feature_group_name} with {max_iterations} iterations...")
        
        # 4. 调用新的多属性去偏函数
        X_deconfounded = inlp_project_multiple_controlled(
            X_base, 
            sens_df, 
            num_iterations=max_iterations
        )
        
        # 5. 将处理好的数据集存入字典
        X_sets_all[feature_group_name] = X_deconfounded

print("\nAll datasets prepared. Total configurations:", len(X_sets_all))
print("Configurations:", list(X_sets_all.keys()))


# --- 接下来，运行您已有的完整评估代码 ---
# 只需将原来的 X_sets 或 X_sets_finetuned 替换为这里的 X_sets_all

n_repeat = 1
model_names = ["Logistic", "RandomForest", "SVM", "XGBoost"]
# 注意：这里我们使用新的 X_sets_all
fair_groups = list(X_sets_all.keys()) 

results_list = []

for model_name in model_names:
    for fg in fair_groups:
        X_data = X_sets_all[fg]
        aucs, accs, f1s, precs, recalls = [], [], [], [], []
        for repeat in range(n_repeat):
            random_state = 42 + repeat
            # 假设 run_vae_and_eval_multi 函数已定义
            metrics = run_vae_and_eval_multi(
                X_data, y, 
                epochs=100, 
                model_name=model_name, 
                n_splits=5, 
                random_state=random_state
            )
            aucs.append(np.mean(metrics['auc_list']))
            accs.append(np.mean(metrics['acc_list']))
            f1s.append(np.mean(metrics['f1_list']))
            precs.append(np.mean(metrics['precision_list']))
            recalls.append(np.mean(metrics['recall_list']))
            
        results_list.append({
            "model": model_name,
            "feature_group": fg,
            "auc_list": aucs, # 保留列表用于后续统计检验
            "auc": np.mean(aucs),
            "acc": np.mean(accs),
            "f1": np.mean(f1s),
            "precision": np.mean(precs),
            "recall": np.mean(recalls),
        })

final_results_df = pd.DataFrame(results_list)

# (可选) 您可以再次运行T检验代码来比较每个去偏组合与基线的差异
# ... (您的T检验代码) ...

print("\n--- Final Results DataFrame ---")
print(final_results_df)

Preparing deconfounded datasets for all combinations...
Processing: Remove-Age with 3 iterations...
Processing: Remove-Edu with 4 iterations...
Processing: Remove-Gender with 8 iterations...
Processing: Remove-Lan with 3 iterations...
Processing: Remove-Age+Edu with 4 iterations...
Processing: Remove-Age+Gender with 8 iterations...
Processing: Remove-Age+Lan with 3 iterations...
Processing: Remove-Edu+Gender with 8 iterations...
Processing: Remove-Edu+Lan with 4 iterations...
Processing: Remove-Gender+Lan with 8 iterations...
Processing: Remove-Age+Edu+Gender with 8 iterations...
Processing: Remove-Age+Edu+Lan with 4 iterations...
Processing: Remove-Age+Gender+Lan with 8 iterations...
Processing: Remove-Edu+Gender+Lan with 8 iterations...
Processing: Remove-Age+Edu+Gender+Lan with 8 iterations...

All datasets prepared. Total configurations: 16
Configurations: ['No-Deconfound', 'Remove-Age', 'Remove-Edu', 'Remove-Gender', 'Remove-Lan', 'Remove-Age+Edu', 'Remove-Age+Gender', 'Remove-Age

In [19]:
final_results_df

,model,feature_group,auc_list,auc,acc,f1,precision,recall
0,Logistic,No-Deconfound,[0.9470109161643695],0.947011,0.871801,0.900293,0.934808,0.868250
1,Logistic,Remove-Age,[0.8750776923534144],0.875078,0.800901,0.841435,0.896894,0.792991
2,Logistic,Remove-Edu,[0.7518308759913008],0.751831,0.697198,0.755352,0.818517,0.701288
3,Logistic,Remove-Gender,[0.9152178533069877],0.915218,0.849987,0.884071,0.911383,0.858431
4,Logistic,Remove-Lan,[0.9026185756320869],0.902619,0.821072,0.857432,0.913495,0.808518
...,...,...,...,...,...,...,...,...
59,XGBoost,Remove-Age+Edu+Gender,[0.6542870272433015],0.654287,0.617028,0.684153,0.759601,0.622740
60,XGBoost,Remove-Age+Edu+Lan,[0.7596255139072954],0.759626,0.691753,0.749959,0.817941,0.693135
61,XGBoost,Remove-Age+Gender+Lan,[0.6660459150788218],0.666046,0.610470,0.673606,0.763021,0.603128
62,XGBoost,Remove-Edu+Gender+Lan,[0.723219897944692],0.723220,0.649222,0.709444,0.790157,0.644901


In [20]:
final_results_df.to_excel("deconfound_evaluation_all_combinations.xlsx", index=False)